In [1]:
import os
from google.colab import userdata


# Définissez vos informations de manière sécurisée
GITHUB_USER = "koubra-gaby"
#GITHUB_TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

# Configuration de l'URL avec authentification
REPO_URL = f"https://@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Cloner le dépôt si ce n'est pas déjà fait
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("Dépôt cloné avec succès !")
else:
    print("Le dépôt existe déjà.")


Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 348, done.
remote: Total 348 (delta 0), reused 0 (delta 0), pack-reused 348 (from 1)
Receiving objects: 100% (348/348), 22.22 MiB | 20.13 MiB/s, done.
Resolving deltas: 100% (164/164), done.
Dépôt cloné avec succès !


In [2]:
# via python & Jupyter
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('update_shu'))

### Entrainer un nouveau model sentencepiece et expandre le tokenizer de nllb

In [1]:
import sentencepiece as spm

# Chemin vers votre fichier de texte brut (ex: corpus.txt)

#input_corpus = "/content/Chadian_arabic-update/Data/full_dataset/arabe_tchad3.txt"
input_corpus = "/kaggle/input/datasets/koubragaby/full-dataset/full_data.txt"
model_prefix = "shu_Latn_sp"

# Entraînement du tokenizer
spm.SentencePieceTrainer.train(
    input=input_corpus,
    model_prefix=model_prefix,
    vocab_size=8000,
    model_type="BPE",
    character_coverage=0.9995,
    byte_fallback=True
)


I0000 00:00:1793368754.079180      50 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /kaggle/input/datasets/koubragaby/full-dataset/full_data.txt
  input_format: 
  model_prefix: shu_Latn_sp
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  

True

### Extraire et nettoyer les news tokens

In [ ]:
# Charger les tokens générés par SentencePiece
new_tokens = []
with open(f"{model_prefix}.vocab", "r", encoding="utf-8") as f:
    for line in f:
        token = line.split("\t")[0]
        # On ignore les tokens de contrôle par défaut de SentencePiece
        if token not in ["<unk>", "<s>", "</s>", "<pad>"]:
            new_tokens.append(token)

print(f"Nombre de tokens extraits : {len(new_tokens)}")


Nombre de tokens extraits : 7997


In [3]:
def vocab_to_subwords(vocab_path, output_path, remove_sentencepiece_prefix=False):
    """
    Convertit un fichier .vocab SentencePiece en fichier .txt
    contenant un token par ligne.

    Args:
        vocab_path: chemin du fichier .vocab
        output_path: chemin du fichier .txt généré
        remove_sentencepiece_prefix:
            Si True, retire le symbole ▁ au début des tokens.
    """

    subwords = []

    with open(vocab_path, "r", encoding="utf-8") as vocab_file:
        for line in vocab_file:
            line = line.strip()

            if not line:
                continue

            # Extraire uniquement le token (avant la tabulation)
            token = line.split("\t")[0]

            # Ignorer les tokens de bytes <0xFC>, etc.
            if token.startswith("<0x") and token.endswith(">"):
                continue

            # Retirer éventuellement le marqueur de début de mot
            if remove_sentencepiece_prefix:
                token = token.removeprefix("▁")

            if token:
                subwords.append(token)

    # Éliminer les doublons tout en conservant l'ordre
    subwords = list(dict.fromkeys(subwords))

    with open(output_path, "w", encoding="utf-8") as output_file:
        output_file.write("\n".join(subwords))

    print(f"{len(subwords)} sous-mots extraits.")
    print(f"Fichier enregistré : {output_path}")

    return subwords

In [4]:
subwords = vocab_to_subwords(
    vocab_path="/home/koubra/Documents/Projet_IA/Chadian arabic update/Data/shu_Latn_sp.vocab",
    output_path="/home/koubra/Documents/Projet_IA/Chadian arabic update/Data/arabic_subwords.txt",
    remove_sentencepiece_prefix=False
)

7744 sous-mots extraits.
Fichier enregistré : /home/koubra/Documents/Projet_IA/Chadian arabic update/Data/arabic_subwords.txt


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_id = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Récupérer le vocabulaire actuel de NLLB
nllb_vocab = tokenizer.get_vocab()

# Filtrer les nouveaux tokens pour ne garder que ceux absents de NLLB
tokens_to_add = [t for t in new_tokens if t not in nllb_vocab]

# 1. Ajouter le code de la nouvelle langue
new_lang_code = "shu_Latn"
tokenizer.add_special_tokens({"additional_special_tokens": [new_lang_code]})

# 4. Mettre à jour les dictionnaires internes de NLLB pour associer le token à une langue
# Cela évite les erreurs lors de la configuration de `src_lang` ou `tgt_lang`

tokenizer.added_tokens_encoder[new_lang_code] = len(tokenizer) - 1
tokenizer.added_tokens_decoder[len(tokenizer) - 1] = new_lang_code

# 2. Ajouter les nouveaux tokens textuels issus de SentencePiece
tokenizer.add_tokens(tokens_to_add)

# 3. Redimensionner les embeddings du modèle
model.resize_token_embeddings(len(tokenizer))

# 4. Sauvegarder le tout
tokenizer.save_pretrained("/content/Chadian_arabic-update/expanded_model_and_tokenizer/nllb-random-custom-tokenizer")
model.save_pretrained("/content/Chadian_arabic-update/expanded_model_and_tokenizer/nllb-random-custom-tokenizer")

print(f"Terminé ! {len(tokens_to_add)} nouveaux tokens et la langue '{new_lang_code}' ont été ajoutés.")


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Terminé ! 5987 nouveaux tokens et la langue 'shu_Latn' ont été ajoutés.


In [ ]:
model.push_to_hub("koubra-gaby/nllb-random-custom-tokenizer")
tokenizer.push_to_hub("koubra-gaby/nllb-random-custom-tokenizer")

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n8p745_/model.safetensors:   0%|          | 27.0kB / 2.48GB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpfxsxnxj4/tokenizer.json:  89%|########9 | 29.8MB / 33.3MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/Koubra-Gaby/nllb-random-custom-tokenizer/commit/9f01110b40e0e17b525fee91ef281976e9593ac5', commit_message='Upload tokenizer', commit_description='', oid='9f01110b40e0e17b525fee91ef281976e9593ac5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Koubra-Gaby/nllb-random-custom-tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='Koubra-Gaby/nllb-random-custom-tokenizer'), pr_revision=None, pr_num=None)

## test d'utilisation

### Nllb

In [12]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M", token=True, src_lang="fra_Latn"
)
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", token=True)

article = "Au commencement, Dieu créa les cieux et la terre."
#inputs = tokenizer(article, return_tensors= "pt")
inputs = tokenizer.tokenize(article)


# translated_tokens = model.generate(
#     **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"), max_length=30
# )z
# tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
inputs

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

['▁Au',
 '▁commen',
 'cement',
 ',',
 '▁Dieu',
 '▁cré',
 'a',
 '▁les',
 '▁cieux',
 '▁et',
 '▁la',
 '▁terre',
 '.']

### Custom nllb

In [11]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "koubra-gaby/nllb-random-custom-tokenizer", token=True, src_lang="shu_Latn"
)
model = AutoModelForSeq2SeqLM.from_pretrained("koubra-gaby/nllb-random-custom-tokenizer", token=True)

article = "Fi l-bidaaya,Allah khalag al-samaawaat wa l-ard."
inputs = tokenizer.tokenize(article)

# translated_tokens = model.generate(
#     **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"), max_length=30
# )
# tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
inputs

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

['▁Fi',
 ' ',
 '▁l',
 '-',
 'bidaaya',
 '▁,',
 'Allah',
 ' khalag',
 ' ',
 '▁al',
 '-',
 'samaawaat',
 ' ',
 '▁wa',
 ' ',
 '▁l',
 '-',
 'ard',
 '.']